# ╔══════════════════════════════════════════════════════════════╗
#  04 · TRANSFORMER FINE-TUNING — SENTIMENT ALLOCINE
#  Pipeline industriel · Optimisé CPU · Portfolio-grade
# ╠══════════════════════════════════════════════════════════════╣
#  Auteur      : DJOKNONE LAURENT et EKWANE Franck
#  Environnement : Dell Latitude 7390 · 16 Go RAM · CPU only
#  Modèles actifs :
#    · camembert-base            (principal — standard FR)
#    · distilbert-multilingual   (comparaison — léger)
#
#  STRATÉGIE CPU :
#    · Sampling stratifié 10% train (16K exemples)
#    · MAX_LENGTH = 128  (P75 Allociné, x4 plus rapide que 256)
#    · 1 epoch   (convergence suffisante sur sous-corpus)
#    · Layer freezing 60% (seules les couches hautes fine-tunées)
#    · Pré-tokenisation en batch (évite overhead à la volée)
#    · Test set complet (20K) — comparable aux baselines
#
#  TEMPS ESTIMÉ :
#    · CamemBERT     : ~35-50 min
#    · DistilBERT    : ~15-20 min
#    · Total         : ~55-70 min
#
#  OUTPUT STANDARD (consommé par notebook 05) :
#  {
#      "model_name"  : str,
#      "short_name"  : str,
#      "y_pred"      : np.ndarray  (int, shape N,),
#      "y_proba"     : np.ndarray  (float, shape N,),
#      "y_true"      : np.ndarray  (int, shape N,),
#      "threshold"   : float,
#      "metrics"     : dict,
#      "model_dir"   : str,
#  }
#
#  INTERDIT dans ce notebook :
#    · Comparaison avec baselines TF-IDF
#    · Graphes avancés (réservés au notebook 05)
#    · Conclusions business
# ╚══════════════════════════════════════════════════════════════╝

In [19]:
# ============================================================
#  VÉRIFICATION DES VERSIONS — EXÉCUTER EN TOUT PREMIER
#  Garantit la compatibilité avant tout import transformers
# ============================================================

import sys
import subprocess

def check_environment() -> bool:
    """
    Vérifie les versions des librairies critiques et la
    compatibilité avec l'API transformers actuelle.
    Retourne True si GPU disponible, False sinon.
    """
    import torch
    import transformers
    import datasets as ds_lib
    from packaging import version

    print("─" * 55)
    print("  VÉRIFICATION ENVIRONNEMENT")
    print("─" * 55)

    libs = {
        "python":       sys.version.split()[0],
        "torch":        torch.__version__,
        "transformers": transformers.__version__,
        "datasets":     ds_lib.__version__,
    }
    for lib, ver in libs.items():
        print(f"   {lib:<15} : {ver}")

    # Vérification argument no_cuda (supprimé en transformers >= 4.36)
    tf_ver = version.parse(transformers.__version__)
    if tf_ver >= version.parse("4.36.0"):
        print(f"\n    transformers {transformers.__version__}")
        print(f"      no_cuda supprimé → on utilise fp16=False sur CPU")
    else:
        print(f"\n     transformers {transformers.__version__} ancien")

    # Vérification eval_strategy vs evaluation_strategy
    if tf_ver >= version.parse("4.38.0"):
        print(f"    eval_strategy disponible (>= 4.38)")
    else:
        print(f"     utiliser evaluation_strategy (< 4.38)")

    # Détection GPU
    cuda_ok = torch.cuda.is_available()
    print(f"\n   CUDA disponible  : {cuda_ok}")
    if cuda_ok:
        print(f"   GPU              : {torch.cuda.get_device_name(0)}")
        print(f"   VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")
    else:
        print(f"   Mode             : CPU uniquement")
        print(f"   Optimisations    : batch=16, max_len=128, 1 epoch, freeze 60%")

    print("─" * 55)
    print(f"   Environnement validé")
    print("─" * 55)
    return cuda_ok


HAS_GPU = check_environment()
print(f"\n  HAS_GPU = {HAS_GPU}  (utilisé par la cellule 3)")

───────────────────────────────────────────────────────
  VÉRIFICATION ENVIRONNEMENT
───────────────────────────────────────────────────────
   python          : 3.10.20
   torch           : 2.11.0+cu130
   transformers    : 5.5.4
   datasets        : 4.8.4

    transformers 5.5.4
      no_cuda supprimé → on utilise fp16=False sur CPU
    eval_strategy disponible (>= 4.38)

   CUDA disponible  : False
   Mode             : CPU uniquement
   Optimisations    : batch=16, max_len=128, 1 epoch, freeze 60%
───────────────────────────────────────────────────────
   Environnement validé
───────────────────────────────────────────────────────

  HAS_GPU = False  (utilisé par la cellule 3)


In [20]:
# ============================================================
#  CELLULE 2/3 — CONFIGURATION + CHARGEMENT DES DONNÉES
#
#  AUTONOME : contient tous ses propres imports.
#  Ne dépend que de la cellule 1 pour HAS_GPU.
#  Si HAS_GPU n'est pas défini, on assume CPU (safe default).
# ============================================================

# ── Imports complets et autonomes ────────────────────────────
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ── Chemins projet ───────────────────────────────────────────
PROJECT_ROOT    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH       = PROJECT_ROOT / "data" / "processed"
MODELS_ROOT     = PROJECT_ROOT / "models" / "transformers"
CHECKPOINTS_DIR = PROJECT_ROOT / "models" / "checkpoints"
REPORTS_DIR     = PROJECT_ROOT / "reports" / "metrics"

for d in [MODELS_ROOT, CHECKPOINTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Seed global ───────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Récupération HAS_GPU avec fallback sécurisé ──────────────
try:
    _ = HAS_GPU   # défini en cellule 1
except NameError:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print(f"     HAS_GPU non trouvé en mémoire → recalculé : {HAS_GPU}")

# ============================================================
#  CONFIGURATION D'ENTRAÎNEMENT
#
#  Chaque paramètre est justifié pour le contexte CPU :
#
#  train_sample_ratio = 0.10
#    → 16 000 exemples sur 160 000.
#    → En dessous de 8% : sous-apprentissage visible.
#    → Au dessus de 20% : > 2h sur CPU pour CamemBERT.
#
#  max_length = 128
#    → Complexité attention = O(n²) : 256→128 divise par 4.
#    → P75 des textes Allociné est à ~120 tokens.
#    → 128 tokens = 75% du signal, 25% du coût de 256.
#
#  num_epochs = 1
#    → Sur 16K exemples, CamemBERT converge en 1 epoch.
#    → 2ème epoch sur CPU : +100% de temps pour <1% de gain.
#
#  batch_size = 16
#    → CPU : plus grand batch = meilleure vectorisation BLAS.
#    → 16 × 128 = 2048 tokens/batch — tient dans 3-4 Go RAM.
#
#  layer_freeze_ratio = 0.60
#    → Gèle les 60% premières couches (feature extractor).
#    → Fine-tune uniquement les couches hautes + tête classif.
#    → Gain ~30% temps, perte <1% sur classification binaire.
#
#  fn_cost = 3.0, fp_cost = 1.0
#    → Aligné avec le notebook 03 (baseline pipeline).
#    → FN 3x plus coûteux que FP pour ce domaine métier.
# ============================================================

CFG = {
    # ── Données ───────────────────────────────────────────────
    "train_sample_ratio": 0.10,
    "val_sample_ratio":   0.25,
    # test = toujours 100% — comparable aux baselines (20K)

    # ── Tokenisation ─────────────────────────────────────────
    "max_length": 128,

    # ── Entraînement ─────────────────────────────────────────
    "num_epochs":                  1,
    "batch_size":                  16,
    "gradient_accumulation_steps": 1,
    "learning_rate":               3e-5,
    "weight_decay":                0.01,
    "warmup_ratio":                0.10,
    "patience":                    1,

    # ── Architecture ─────────────────────────────────────────
    "num_labels":         2,
    "layer_freeze_ratio": 0.60,

    # ── Seuil métier ─────────────────────────────────────────
    "fn_cost": 3.0,
    "fp_cost": 1.0,
}

# ============================================================
#  REGISTRE DES MODÈLES
# ============================================================

MODEL_REGISTRY = {

    "camembert-base": {
        "hf_name":    "camembert-base",
        "short_name": "CamemBERT-base",
        "rationale": (
            "Standard de facto du NLP français. "
            "Architecture RoBERTa entraînée sur 138 Go OSCAR (corpus FR). "
            "Référence incontournable pour tout projet NLP français."
        ),
    },

    "distilbert-multilingual": {
        "hf_name":    "distilbert-base-multilingual-cased",
        "short_name": "DistilBERT-mlt",
        "rationale": (
            "Version distillée de mBERT. 66M params, 40% plus léger. "
            "Multilingue (104 langues). Comparaison pertinente : "
            "spécialisé FR (CamemBERT) vs généraliste léger (DistilBERT)."
        ),
    },
}

# Modèles actifs pour cet environnement
# Sur CPU : 2 modèles max pour des temps raisonnables
ACTIVE_MODELS = ["camembert-base", "distilbert-multilingual"]

# ============================================================
#  CHARGEMENT DES DONNÉES
# ============================================================

def load_split_data(
    data_path:  Path,
    split:      str,
    ratio:      float,
    text_col:   str = "text_transformer",
    seed:       int = SEED,
) -> tuple[list, list]:
    """
    Charge un split Parquet avec sous-échantillonnage stratifié.

    Paramètres
    ----------
    data_path : chemin vers data/processed/
    split     : "train", "validation" ou "test"
    ratio     : fraction à conserver (1.0 = complet)
    text_col  : colonne de texte à utiliser
    seed      : graine pour reproductibilité

    Retourne
    --------
    texts  : list[str] — textes nettoyés
    labels : list[int] — labels 0/1

    Notes
    -----
    text_transformer = nettoyage minimal (HTML, URLs, emojis).
    NE PAS utiliser text_classical (lowercase, sans ponctuation) :
    CamemBERT a été pré-entraîné sur du texte naturel non altéré.
    Enlever les majuscules et la ponctuation dégraderait les perfs.

    Stratification : garantit 50/50 pos/neg dans l'échantillon,
    même sur un sous-ensemble. Un échantillon non stratifié
    pourrait biaiser l'entraînement si le hasard déséquilibre.
    """
    path = data_path / f"{split}.parquet"

    if not path.exists():
        raise FileNotFoundError(
            f"\n   Fichier manquant : {path}"
            f"\n   Lance d'abord :"
            f"\n   python src/data/download_dataset.py"
            f"\n   python src/data/preprocess.py"
        )

    df = pd.read_parquet(path)

    # Validation des colonnes critiques
    missing_cols = [c for c in [text_col, "label"] if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Colonnes manquantes dans '{split}' : {missing_cols}"
            f"\nColonnes disponibles : {list(df.columns)}"
        )

    texts  = df[text_col].fillna("").astype(str).tolist()
    labels = df["label"].astype(int).tolist()

    # Sous-échantillonnage stratifié uniquement si ratio < 1.0
    if ratio < 1.0:
        _, texts, _, labels = train_test_split(
            texts, labels,
            test_size=ratio,
            stratify=labels,
            random_state=seed,
        )

    pos_rate = np.mean(labels)
    status   = "complet" if ratio >= 1.0 else f"sampled {ratio:.0%}"

    print(
        f"   [{split:>10}]  ({status:<12})  "
        f"n={len(texts):>6,}  |  "
        f"pos={pos_rate:.1%}  neg={1-pos_rate:.1%}"
    )
    return texts, labels


print("─" * 55)
print("  CHARGEMENT DES DONNÉES")
print("─" * 55)

texts_train,  labels_train  = load_split_data(DATA_PATH, "train",      CFG["train_sample_ratio"])
texts_val,    labels_val    = load_split_data(DATA_PATH, "validation",  CFG["val_sample_ratio"])
texts_test,   labels_test   = load_split_data(DATA_PATH, "test",        1.0)

# ── Assertions de cohérence ───────────────────────────────────
assert len(texts_train)  == len(labels_train),  "Mismatch train"
assert len(texts_val)    == len(labels_val),    "Mismatch val"
assert len(texts_test)   == len(labels_test),   "Mismatch test"
assert len(texts_test)   == 20_000, (
    f"Test set incomplet : {len(texts_test)} ex. (attendu 20 000)"
)

print(f"\n   Données prêtes")
print(f"     Colonne        : 'text_transformer'")
print(f"     Test set       : {len(texts_test):,} ex. (complet — comparable aux baselines)")
print(f"\n  CFG résumé :")
print(f"     train_ratio    : {CFG['train_sample_ratio']:.0%}  ({len(texts_train):,} ex.)")
print(f"     val_ratio      : {CFG['val_sample_ratio']:.0%}  ({len(texts_val):,} ex.)")
print(f"     max_length     : {CFG['max_length']}")
print(f"     epochs         : {CFG['num_epochs']}")
print(f"     batch_size     : {CFG['batch_size']}")
print(f"     learning_rate  : {CFG['learning_rate']}")
print(f"     fn_cost/fp_cost: {CFG['fn_cost']}/{CFG['fp_cost']}")
print(f"     freeze_ratio   : {CFG['layer_freeze_ratio']:.0%}")

───────────────────────────────────────────────────────
  CHARGEMENT DES DONNÉES
───────────────────────────────────────────────────────
   [     train]  (sampled 10% )  n=16,000  |  pos=50.4%  neg=49.6%
   [validation]  (sampled 25% )  n= 5,000  |  pos=49.0%  neg=51.0%
   [      test]  (complet     )  n=20,000  |  pos=48.0%  neg=52.0%

   Données prêtes
     Colonne        : 'text_transformer'
     Test set       : 20,000 ex. (complet — comparable aux baselines)

  CFG résumé :
     train_ratio    : 10%  (16,000 ex.)
     val_ratio      : 25%  (5,000 ex.)
     max_length     : 128
     epochs         : 1
     batch_size     : 16
     learning_rate  : 3e-05
     fn_cost/fp_cost: 3.0/1.0
     freeze_ratio   : 60%


In [21]:
# ============================================================
#   PIPELINE COMPLET AUTONOME
#
#  CONTIENT : tous les imports, toutes les fonctions,
#             la boucle d'entraînement, le registre.
#
#  DÉPEND DE : cellule 2 uniquement (pour CFG, MODEL_REGISTRY,
#              ACTIVE_MODELS, texts_*, labels_*, SEED, HAS_GPU,
#              PROJECT_ROOT, MODELS_ROOT, CHECKPOINTS_DIR).
#
#  STRATÉGIE : tous les imports sont redéclarés ici.
#  Si une variable de cellule 2 est manquante, on lève
#  une erreur explicite avant de lancer le pipeline.
# ============================================================

# ── Imports complets — redéclarés pour autonomie totale ──────
import gc
import json
import os
import sys
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from scipy.special import softmax

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    EvalPrediction,
    set_seed as hf_set_seed,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "4"
os.environ["MKL_NUM_THREADS"]        = "4"

# ── Vérification des variables de cellule 2 ──────────────────
_required = {
    "CFG":           "Configuration d'entraînement",
    "MODEL_REGISTRY":"Registre des modèles",
    "ACTIVE_MODELS": "Liste des modèles actifs",
    "texts_train":   "Textes d'entraînement",
    "labels_train":  "Labels d'entraînement",
    "texts_val":     "Textes de validation",
    "labels_val":    "Labels de validation",
    "texts_test":    "Textes de test",
    "labels_test":   "Labels de test",
    "SEED":          "Graine de reproductibilité",
    "PROJECT_ROOT":  "Racine du projet",
    "MODELS_ROOT":   "Dossier des modèles",
    "CHECKPOINTS_DIR": "Dossier des checkpoints",
}

_missing = []
for _var, _desc in _required.items():
    if _var not in dir():
        _missing.append(f"     {_var:<20} ({_desc})")

if _missing:
    raise RuntimeError(
        "\n\nVARIABLES MANQUANTES — Exécute d'abord la cellule 2 :\n"
        + "\n".join(_missing)
    )

# Récupération HAS_GPU avec fallback
try:
    _ = HAS_GPU
except NameError:
    HAS_GPU = torch.cuda.is_available()

# Reproductibilité complète
torch.manual_seed(SEED)
hf_set_seed(SEED)

DEVICE = torch.device("cuda" if HAS_GPU else "cpu")

print(f"    Tous les imports OK")
print(f"  Device : {DEVICE}")
print(f"  Modèles actifs : {ACTIVE_MODELS}")


# ════════════════════════════════════════════════════════════
#  BLOC A — DATASET PYTORCH
#  Pré-tokenisation batch dans __init__ :
#  plus efficace que tokenisation à la volée sur CPU
#  (le tokenizer Rust est optimisé pour les gros batches).
# ════════════════════════════════════════════════════════════

class SentimentDataset(Dataset):
    """
    Dataset pré-tokenisé pour le Trainer HuggingFace.

    La tokenisation en batch dans __init__ évite de retokeniser
    chaque texte individuellement à chaque accès __getitem__.
    Sur CPU : gain ~15-20% sur le temps d'entraînement total.
    Coût RAM : ~50 Mo pour 16K textes × 128 tokens (négligeable).
    """

    def __init__(
        self,
        texts:      list,
        labels:     list,
        tokenizer,
        max_length: int,
        split_name: str = "",
    ):
        assert len(texts) == len(labels), (
            f"Mismatch : {len(texts)} textes vs {len(labels)} labels"
        )

        print(
            f"   [{split_name:<6}] Tokenisation de {len(texts):,} textes ...",
            end=" ", flush=True,
        )
        t0 = time.time()

        # Tokenisation vectorisée en une seule passe
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        # Stockage des tenseurs pré-calculés
        self.input_ids      = enc["input_ids"]        # shape (N, max_length)
        self.attention_mask = enc["attention_mask"]   # shape (N, max_length)
        self.labels         = torch.tensor(labels, dtype=torch.long)  # shape (N,)

        elapsed = time.time() - t0
        print(f"  ({elapsed:.1f}s) | RAM≈{self.input_ids.nbytes / 1e6:.0f} Mo")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        """
        Retourne exactement les 3 clés attendues par le Trainer HF.
        Accès O(1) sur les tenseurs pré-calculés.
        """
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


# ════════════════════════════════════════════════════════════
#  BLOC B — MÉTRIQUES & SEUIL MÉTIER
# ════════════════════════════════════════════════════════════

def compute_metrics_hf(eval_pred: EvalPrediction) -> dict:
    """
    Métriques pour le Trainer HuggingFace.
    Appelée automatiquement à chaque fin d'epoch.
    F1 = critère de sélection du meilleur checkpoint.

    Paramètres
    ----------
    eval_pred : EvalPrediction — namedtuple(predictions, label_ids)
    """
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    # softmax : logits → probabilités (somme = 1 par exemple)
    probas = softmax(logits, axis=1)[:, 1]
    # argmax : classe avec le logit le plus élevé
    preds  = np.argmax(logits, axis=1)

    return {
        "f1":        round(float(f1_score(labels, preds, zero_division=0)), 4),
        "accuracy":  round(float(accuracy_score(labels, preds)), 4),
        "precision": round(float(precision_score(labels, preds, zero_division=0)), 4),
        "recall":    round(float(recall_score(labels, preds, zero_division=0)), 4),
        "roc_auc":   round(float(roc_auc_score(labels, probas)), 4),
    }


def optimize_threshold(
    y_proba:  np.ndarray,
    y_true:   np.ndarray,
    fn_cost:  float = 3.0,
    fp_cost:  float = 1.0,
    n_points: int   = 300,
) -> tuple:
    """
    Cherche le seuil minimisant le coût métier sur la VALIDATION.

    Coût : fn_cost × FN + fp_cost × FP
    Aligné avec notebook 03 (fn_cost=3, fp_cost=1).

    RÈGLE ABSOLUE : jamais appeler avec les données de test.
    Ce serait du data leakage → biais d'optimisme sur le test.

    Paramètres
    ----------
    y_proba  : probabilités classe positive (validation)
    y_true   : labels vrais (validation)
    fn_cost  : poids des faux négatifs
    fp_cost  : poids des faux positifs
    n_points : granularité de la recherche

    Retourne
    --------
    best_t       : float — seuil optimal
    df_thresholds: DataFrame — métriques par seuil (pour notebook 05)
    """
    thresholds = np.linspace(0.01, 0.99, n_points)
    records    = []

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        records.append({
            "threshold": round(float(t), 5),
            "cost":      fn_cost * fn + fp_cost * fp,
            "fn":        fn,
            "fp":        fp,
            "f1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
            "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
            "recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        })

    df     = pd.DataFrame(records)
    best_t = float(df.loc[df["cost"].idxmin(), "threshold"])

    print(f"   Seuil optimal : {best_t:.4f}  (fn_cost={fn_cost}, fp_cost={fp_cost})")
    print(f"   F1 au seuil  : {df.loc[df['cost'].idxmin(), 'f1']:.4f}")
    print(f"   FN / FP      : {int(df.loc[df['cost'].idxmin(), 'fn'])} / "
          f"{int(df.loc[df['cost'].idxmin(), 'fp'])}")

    return best_t, df


# ════════════════════════════════════════════════════════════
#  BLOC C — 9 FONCTIONS DU PIPELINE
#  Chaque fonction a une responsabilité unique.
#  Signatures stables, aucune variable globale implicite.
# ════════════════════════════════════════════════════════════

def load_tokenizer(hf_name: str):
    """
    Charge le tokenizer depuis HuggingFace Hub.
    use_fast=True : implémentation Rust, 10-100x plus rapide.
    """
    tok = AutoTokenizer.from_pretrained(hf_name, use_fast=True)
    print(f"   Type    : {type(tok).__name__}")
    print(f"   Vocab   : {tok.vocab_size:,}")
    return tok


def tokenize_dataset(
    texts_train:  list, labels_train: list,
    texts_val:    list, labels_val:   list,
    texts_test:   list, labels_test:  list,
    tokenizer,
    max_length:   int,
) -> tuple:
    """
    Construit les 3 SentimentDataset pré-tokenisés.
    La tokenisation batch est faite une seule fois dans __init__.
    """
    print(f"   max_length = {max_length}")
    ds_train = SentimentDataset(texts_train, labels_train, tokenizer, max_length, "train")
    ds_val   = SentimentDataset(texts_val,   labels_val,   tokenizer, max_length, "val")
    ds_test  = SentimentDataset(texts_test,  labels_test,  tokenizer, max_length, "test")
    return ds_train, ds_val, ds_test


def build_model(
    hf_name:            str,
    num_labels:         int,
    layer_freeze_ratio: float,
) -> tuple:
    """
    Charge le modèle pré-entraîné et applique le layer freezing.

    Les warnings UNEXPECTED/MISSING sont normaux et attendus :
    on remplace la tête MLM du modèle original par une tête de
    classification à 2 labels. Ces poids sont réinitialisés.

    Layer freezing : gèle les `layer_freeze_ratio`% premières couches.
    Le modèle a déjà appris des représentations génériques de qualité.
    On ne fine-tune que les couches hautes + la nouvelle tête.
    Gain : ~30% de temps, perte : <1% sur classification binaire.

    Retourne
    --------
    model : modèle avec gradients configurés
    info  : dict de métriques sur les paramètres
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_name,
        num_labels=num_labels,
        # ignore_mismatched_sizes : permet de remplacer la tête
        # sans erreur même si les dimensions diffèrent
        ignore_mismatched_sizes=True,
    )

    n_total  = sum(p.numel() for p in model.parameters())
    all_pms  = list(model.named_parameters())
    n_freeze = int(len(all_pms) * layer_freeze_ratio)

    # Gel des premières couches
    for i, (_, param) in enumerate(all_pms):
        if i < n_freeze:
            param.requires_grad = False

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    info = {
        "n_total":     n_total,
        "n_trainable": n_trainable,
        "n_frozen":    n_freeze,
        "n_layers":    len(all_pms),
    }

    print(f"   Params totaux   : {n_total:,}")
    print(f"   Params entraîn. : {n_trainable:,} ({n_trainable/n_total:.0%})")
    print(f"   Couches gelées  : {n_freeze}/{len(all_pms)} ({layer_freeze_ratio:.0%})")
    print(f"   Gain estimé     : ~30% de temps vs fine-tuning complet")

    return model, info


def build_training_args(
    output_dir: Path,
    cfg:        dict,
    has_gpu:    bool = False,
) -> TrainingArguments:
    """
    Construit les TrainingArguments compatibles transformers >= 4.36.

    Changements importants vs versions anciennes :
    ─ no_cuda supprimé en >= 4.36 → on utilise fp16=False sur CPU
    ─ evaluation_strategy renommé eval_strategy en >= 4.38
    ─ use_cpu supprimé → PyTorch détecte automatiquement le device

    On utilise uniquement les arguments stables et documentés.

    Paramètres
    ----------
    output_dir : dossier de sauvegarde des checkpoints
    cfg        : dict CFG avec les hyperparamètres
    has_gpu    : True si GPU disponible
    """
    return TrainingArguments(
        output_dir=str(output_dir),

        # ── Entraînement ──────────────────────────────────────
        num_train_epochs               = cfg["num_epochs"],
        per_device_train_batch_size    = cfg["batch_size"],
        # eval batch 2x : pas de gradient → mémoire réduite
        per_device_eval_batch_size     = cfg["batch_size"] * 2,
        gradient_accumulation_steps    = cfg["gradient_accumulation_steps"],
        learning_rate                  = cfg["learning_rate"],
        weight_decay                   = cfg["weight_decay"],
        warmup_ratio                   = cfg["warmup_ratio"],
        lr_scheduler_type              = "linear",
        max_grad_norm                  = 1.0,

        # ── Évaluation & checkpoint ───────────────────────────
        eval_strategy                  = "epoch",   # >= 4.38
        save_strategy                  = "epoch",
        load_best_model_at_end         = True,
        metric_for_best_model          = "f1",
        greater_is_better              = True,
        # 1 seul checkpoint : CamemBERT pèse ~450 Mo
        save_total_limit               = 1,

        # ── CPU / GPU ─────────────────────────────────────────
        # fp16 : demi-précision uniquement sur GPU
        # Float16 non supporté nativement sur CPU Intel classique
        fp16                           = has_gpu,
        # workers=0 : le GIL Python rend le multiprocessing inutile
        # quand les tenseurs sont déjà en mémoire (pré-tokenisés)
        dataloader_num_workers         = 0,
        # pin_memory : transfert DMA CPU→GPU uniquement
        dataloader_pin_memory          = has_gpu,

        # ── Logs & reproductibilité ───────────────────────────
        seed                           = SEED,
        data_seed                      = SEED,
        logging_strategy               = "steps",
        logging_steps                  = 20,
        report_to                      = "none",
        disable_tqdm                   = False,
        push_to_hub                    = False,
    )


def build_trainer(
    model:         object,
    training_args: TrainingArguments,
    ds_train:      SentimentDataset,
    ds_val:        SentimentDataset,
    patience:      int,
) -> Trainer:
    """
    Construit le Trainer avec EarlyStoppingCallback.
    Arrête si F1 ne progresse pas de > 0.001 sur `patience` epochs.
    """
    return Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = ds_train,
        eval_dataset    = ds_val,
        compute_metrics = compute_metrics_hf,
        callbacks       = [
            EarlyStoppingCallback(
                early_stopping_patience  = patience,
                early_stopping_threshold = 0.001,
            )
        ],
    )


def train_model(trainer: Trainer) -> tuple:
    """
    Lance l'entraînement.

    Retourne
    --------
    train_result : objet TrainOutput HuggingFace
    history      : list[dict] — métriques validation par epoch
    t_sec        : float — durée en secondes
    """
    t0     = time.time()
    result = trainer.train()
    t_sec  = round(time.time() - t0, 1)

    # Extraction de l'historique de validation
    history = [
        {k: round(v, 4) if isinstance(v, float) else v
         for k, v in log.items()}
        for log in trainer.state.log_history
        if "eval_f1" in log
    ]

    print(f"   Durée        : {t_sec:.1f}s  ({t_sec/60:.1f} min)")
    print(f"   Steps totaux : {result.global_step}")
    print(f"   Loss finale  : {result.training_loss:.4f}")

    if history:
        print(f"   Historique validation :")
        for h in history:
            print(
                f"     epoch={h.get('epoch','?')}  "
                f"F1={h.get('eval_f1','?')}  "
                f"loss={h.get('eval_loss','?')}"
            )

    return result, history, t_sec


def evaluate_model(
    trainer:     Trainer,
    ds_val:      SentimentDataset,
    ds_test:     SentimentDataset,
    labels_val:  list,
    labels_test: list,
    fn_cost:     float,
    fp_cost:     float,
) -> dict:
    """
    Inférence val → seuil métier, puis test → métriques finales.

    ORDRE OBLIGATOIRE :
    1. Inférer sur validation
    2. Optimiser le seuil sur validation
    3. Inférer sur test
    4. Appliquer le seuil trouvé en (2) sur les probas de (3)

    Jamais : optimiser le seuil sur le test → data leakage.
    """
    # ── Inférence validation ─────────────────────────────────
    print("   Inférence validation ...", end=" ", flush=True)
    t0          = time.time()
    val_out     = trainer.predict(ds_val)
    probas_val  = softmax(val_out.predictions, axis=1)[:, 1]
    labels_val_np = np.array(labels_val)
    print(f"({time.time()-t0:.1f}s)")

    # ── Seuil métier sur validation ───────────────────────────
    best_t, threshold_df = optimize_threshold(
        y_proba = probas_val,
        y_true  = labels_val_np,
        fn_cost = fn_cost,
        fp_cost = fp_cost,
    )

    # ── Inférence test ────────────────────────────────────────
    print("   Inférence test ...", end=" ", flush=True)
    t0             = time.time()
    test_out       = trainer.predict(ds_test)
    probas_test    = softmax(test_out.predictions, axis=1)[:, 1]
    labels_test_np = np.array(labels_test)
    print(f"({time.time()-t0:.1f}s)")

    # ── Prédictions avec seuil optimal ───────────────────────
    y_pred = (probas_test >= best_t).astype(int)

    return {
        "y_pred":       y_pred,
        "y_proba":      probas_test,
        "y_true":       labels_test_np,
        "threshold":    best_t,
        "threshold_df": threshold_df,
    }


def compute_final_metrics(
    y_true:      np.ndarray,
    y_pred:      np.ndarray,
    y_proba:     np.ndarray,
    threshold:   float,
    train_result,
    train_time:  float,
    model_info:  dict,
    cfg:         dict,
    n_train:     int,
    n_val:       int,
    n_test:      int,
) -> dict:
    """
    Calcule et structure toutes les métriques finales sur le test set.
    Affiche le classification_report complet.
    """
    metrics = {
        "f1":                 round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "accuracy":           round(float(accuracy_score(y_true, y_pred)), 4),
        "precision":          round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall":             round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "roc_auc":            round(float(roc_auc_score(y_true, y_proba)), 4),
        "threshold":          round(threshold, 6),
        "train_time_s":       train_time,
        "global_step":        train_result.global_step,
        "training_loss":      round(float(train_result.training_loss), 6),
        "n_params_total":     model_info["n_total"],
        "n_params_trainable": model_info["n_trainable"],
        "sampling": {
            "train_ratio":  cfg["train_sample_ratio"],
            "val_ratio":    cfg["val_sample_ratio"],
            "n_train":      n_train,
            "n_val":        n_val,
            "n_test":       n_test,
            "max_length":   cfg["max_length"],
            "layer_freeze": f"{model_info['n_frozen']}/{model_info['n_layers']}",
        },
    }

    print(f"\n{'─'*55}")
    print(classification_report(
        y_true, y_pred,
        target_names=["Négatif", "Positif"],
        digits=4,
    ))
    print(f"   ROC AUC  : {metrics['roc_auc']:.4f}")
    print(f"   Seuil    : {metrics['threshold']:.4f}")
    print(f"   Temps    : {train_time/60:.1f} min")

    return metrics


def save_artifacts(
    model_dir:     Path,
    trainer:       Trainer,
    tokenizer,
    eval_output:   dict,
    metrics:       dict,
    train_history: list,
    model_key:     str,
    short_name:    str,
    hf_name:       str,
    cfg:           dict,
    model_info:    dict,
) -> None:
    """
    Sauvegarde tous les artefacts nécessaires au notebook 05 et à l'API.

    Structure produite :
    models/transformers/{model_key}/
      ├── model.safetensors   (poids du modèle)
      ├── tokenizer files...  (tokenizer HF)
      ├── config.json         (config complète reproductible)
      ├── threshold.json      (seuil seul — accès rapide API)
      ├── threshold_analysis.parquet (courbe coût/seuil)
      ├── y_pred.npy          (prédictions test)
      ├── y_proba.npy         (probabilités test)
      └── y_true.npy          (labels test — référence)
    """
    model_dir.mkdir(parents=True, exist_ok=True)

    # Modèle + tokenizer au format HuggingFace natif
    # Rechargeable : AutoModel.from_pretrained(str(model_dir))
    trainer.save_model(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
    print(f"     Modèle + tokenizer")

    # Arrays numpy pour notebook 05
    np.save(model_dir / "y_pred.npy",  eval_output["y_pred"])
    np.save(model_dir / "y_proba.npy", eval_output["y_proba"])
    np.save(model_dir / "y_true.npy",  eval_output["y_true"])
    print(f"     y_pred / y_proba / y_true (.npy)")

    # Courbe coût/seuil pour notebook 05
    eval_output["threshold_df"].to_parquet(
        model_dir / "threshold_analysis.parquet", index=False
    )
    print(f"     threshold_analysis.parquet")

    # Config JSON complète
    artifact = {
        "model_key":     model_key,
        "short_name":    short_name,
        "hf_name":       hf_name,
        "threshold":     round(eval_output["threshold"], 6),
        "max_length":    cfg["max_length"],
        "metrics":       metrics,
        "train_history": train_history,
        "cpu_optimized": True,
        "layer_freeze":  f"{model_info['n_frozen']}/{model_info['n_layers']}",
        "training_args": {
            "num_epochs":    cfg["num_epochs"],
            "batch_size":    cfg["batch_size"],
            "learning_rate": cfg["learning_rate"],
            "weight_decay":  cfg["weight_decay"],
            "warmup_ratio":  cfg["warmup_ratio"],
            "seed":          SEED,
        },
    }
    with open(model_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(artifact, f, indent=2, ensure_ascii=False)

    # Seuil seul pour l'API de production
    with open(model_dir / "threshold.json", "w") as f:
        json.dump({
            "threshold": round(eval_output["threshold"], 6),
            "model_key": model_key,
        }, f)

    print(f"     config.json + threshold.json")


print("    Toutes les fonctions définies (A, B, C)")
print("  Lancement de la boucle dans 3 secondes...")
time.sleep(1)


# ════════════════════════════════════════════════════════════
#  BLOC D — BOUCLE PRINCIPALE D'ENTRAÎNEMENT
# ════════════════════════════════════════════════════════════

TRANSFORMER_RESULTS: dict = {}
FAILED_MODELS:       list = []

t_global = time.time()

print(f"\n{'═'*62}")
print(f"  LANCEMENT — {len(ACTIVE_MODELS)} modèle(s)")
print(f"  Ordre   : {' → '.join(ACTIVE_MODELS)}")
print(f"  Device  : {DEVICE}")
print(f"  HAS_GPU : {HAS_GPU}")
print(f"{'═'*62}")

for idx, model_key in enumerate(ACTIVE_MODELS, 1):

    cfg_m      = MODEL_REGISTRY[model_key]
    hf_name    = cfg_m["hf_name"]
    short_name = cfg_m["short_name"]
    model_dir  = MODELS_ROOT / model_key
    ckpt_dir   = CHECKPOINTS_DIR / model_key

    print(f"\n\n{'═'*62}")
    print(f"  [{idx}/{len(ACTIVE_MODELS)}] {short_name}")
    print(f"  HF      : {hf_name}")
    print(f"  Output  : models/transformers/{model_key}/")
    print(f"{'═'*62}")

    try:
        t_model = time.time()

        # ── [1/9] Tokenizer ───────────────────────────────────
        print(f"\n[1/9] Tokenizer ...")
        tokenizer = load_tokenizer(hf_name)

        # ── [2/9] Datasets pré-tokenisés ──────────────────────
        print(f"\n[2/9] Datasets pré-tokenisés ...")
        ds_train, ds_val, ds_test = tokenize_dataset(
            texts_train, labels_train,
            texts_val,   labels_val,
            texts_test,  labels_test,
            tokenizer,
            CFG["max_length"],
        )

        # ── [3/9] Modèle + layer freezing ─────────────────────
        print(f"\n[3/9] Modèle + layer freezing ({CFG['layer_freeze_ratio']:.0%}) ...")
        model, model_info = build_model(
            hf_name            = hf_name,
            num_labels         = CFG["num_labels"],
            layer_freeze_ratio = CFG["layer_freeze_ratio"],
        )

        # ── [4/9] TrainingArguments ───────────────────────────
        print(f"\n[4/9] TrainingArguments ...")
        training_args = build_training_args(
            output_dir = ckpt_dir,
            cfg        = CFG,
            has_gpu    = HAS_GPU,
        )
        print(
            f"   fp16={training_args.fp16}  "
            f"lr={training_args.learning_rate}  "
            f"epochs={int(training_args.num_train_epochs)}  "
            f"batch={training_args.per_device_train_batch_size}"
        )

        # ── [5/9] Trainer ─────────────────────────────────────
        print(f"\n[5/9] Trainer + EarlyStopping ...")
        trainer = build_trainer(
            model         = model,
            training_args = training_args,
            ds_train      = ds_train,
            ds_val        = ds_val,
            patience      = CFG["patience"],
        )

        # ── [6/9] Entraînement ────────────────────────────────
        print(f"\n[6/9] Entraînement ...")
        train_result, history, t_train = train_model(trainer)

        # ── [7/9] Évaluation ──────────────────────────────────
        print(f"\n[7/9] Évaluation (val → seuil, test → métriques) ...")
        eval_output = evaluate_model(
            trainer     = trainer,
            ds_val      = ds_val,
            ds_test     = ds_test,
            labels_val  = labels_val,
            labels_test = labels_test,
            fn_cost     = CFG["fn_cost"],
            fp_cost     = CFG["fp_cost"],
        )

        # ── [8/9] Métriques finales ───────────────────────────
        print(f"\n[8/9] Métriques finales (test — {len(labels_test):,} ex.) ...")
        metrics = compute_final_metrics(
            y_true      = eval_output["y_true"],
            y_pred      = eval_output["y_pred"],
            y_proba     = eval_output["y_proba"],
            threshold   = eval_output["threshold"],
            train_result= train_result,
            train_time  = t_train,
            model_info  = model_info,
            cfg         = CFG,
            n_train     = len(texts_train),
            n_val       = len(texts_val),
            n_test      = len(texts_test),
        )

        # ── [9/9] Sauvegarde ──────────────────────────────────
        print(f"\n[9/9] Sauvegarde artefacts ...")
        save_artifacts(
            model_dir     = model_dir,
            trainer       = trainer,
            tokenizer     = tokenizer,
            eval_output   = eval_output,
            metrics       = metrics,
            train_history = history,
            model_key     = model_key,
            short_name    = short_name,
            hf_name       = hf_name,
            cfg           = CFG,
            model_info    = model_info,
        )

        # ── Stockage résultat standardisé ─────────────────────
        TRANSFORMER_RESULTS[model_key] = {
            "model_name":    model_key,
            "short_name":    short_name,
            "hf_name":       hf_name,
            "y_pred":        eval_output["y_pred"],
            "y_proba":       eval_output["y_proba"],
            "y_true":        eval_output["y_true"],
            "threshold":     eval_output["threshold"],
            "metrics":       metrics,
            "train_history": history,
            "model_dir":     str(model_dir),
        }

        t_elapsed = round(time.time() - t_model, 1)
        print(f"\n    {short_name} — terminé en {t_elapsed:.1f}s ({t_elapsed/60:.1f} min)")
        print(f"     F1={metrics['f1']:.4f}  "
              f"AUC={metrics['roc_auc']:.4f}  "
              f"Seuil={metrics['threshold']:.4f}")

    except Exception as exc:
        # Robustesse : on log le traceback complet et on continue
        print(f"\n    ERREUR — {short_name}")
        print(f"     Type    : {type(exc).__name__}")
        print(f"     Message : {str(exc)[:500]}")
        print(f"     Traceback complet :")
        traceback.print_exc()
        FAILED_MODELS.append(model_key)

    finally:
        # Libération mémoire dans TOUS les cas
        # Critique : sans cette étape, le 2ème modèle peut provoquer OOM
        _cleanup_vars = [
            "model", "trainer", "tokenizer",
            "ds_train", "ds_val", "ds_test",
            "val_out", "test_out", "training_args",
            "train_result", "eval_output", "history",
        ]
        for _v in _cleanup_vars:
            if _v in locals():
                del locals()[_v]
        gc.collect()


# ════════════════════════════════════════════════════════════
#  BLOC E — RÉSUMÉ + EXPORT REGISTRE NOTEBOOK 05
# ════════════════════════════════════════════════════════════

t_total = round(time.time() - t_global, 1)

print(f"\n\n{'═'*62}")
print(f"  BOUCLE TERMINÉE")
print(f"  Durée totale : {t_total:.1f}s  ({t_total/60:.1f} min)")
print(f"  Réussis      : {len(TRANSFORMER_RESULTS)}/{len(ACTIVE_MODELS)}")
if FAILED_MODELS:
    print(f"  Échoués      : {FAILED_MODELS}")
print(f"{'═'*62}")

if not TRANSFORMER_RESULTS:
    print("\n   Aucun modèle réussi. Relire les erreurs ci-dessus.")

else:
    # ── Résumé des performances ───────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  RÉSUMÉ DES PERFORMANCES (test set)")
    print(f"{'─'*55}")

    rows = []
    for key, res in TRANSFORMER_RESULTS.items():
        m = res["metrics"]
        rows.append({
            "Modèle":       res["short_name"],
            "F1":           m["f1"],
            "Accuracy":     m["accuracy"],
            "Precision":    m["precision"],
            "Recall":       m["recall"],
            "ROC_AUC":      m["roc_auc"],
            "Seuil":        m["threshold"],
            "Temps_min":    round(m["train_time_s"] / 60, 1),
        })

    df_summary = pd.DataFrame(rows).sort_values("F1", ascending=False)
    print(df_summary.to_string(index=False))

    # ── Validation de cohérence ───────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  VALIDATION DE COHÉRENCE")
    print(f"{'─'*55}")

    for key, res in TRANSFORMER_RESULTS.items():
        m    = res["metrics"]
        msgs = []
        if m["f1"]      < 0.75: msgs.append(f"F1={m['f1']:.4f} trop bas (attendu >0.80)")
        if m["roc_auc"] < 0.85: msgs.append(f"ROC AUC={m['roc_auc']:.4f} trop bas")
        if not (0.10 <= m["threshold"] <= 0.90):
            msgs.append(f"Seuil={m['threshold']:.4f} hors plage [0.10, 0.90]")

        if msgs:
            for msg in msgs:
                print(f"      {res['short_name']} : {msg}")
        else:
            print(f"     {res['short_name']} — métriques cohérentes")

    # ── Export registre pour notebook 05 ─────────────────────
    registry = {}
    for key, res in TRANSFORMER_RESULTS.items():
        mp = Path(res["model_dir"])
        registry[key] = {
            "short_name": res["short_name"],
            "hf_name":    res["hf_name"],
            "model_dir":  res["model_dir"],
            "threshold":  res["threshold"],
            "metrics":    res["metrics"],
            "artifacts": {
                "y_pred":             str(mp / "y_pred.npy"),
                "y_proba":            str(mp / "y_proba.npy"),
                "y_true":             str(mp / "y_true.npy"),
                "config":             str(mp / "config.json"),
                "threshold":          str(mp / "threshold.json"),
                "threshold_analysis": str(mp / "threshold_analysis.parquet"),
            },
        }

    registry_path = MODELS_ROOT / "transformer_registry.json"
    with open(registry_path, "w", encoding="utf-8") as f:
        json.dump(registry, f, indent=2, ensure_ascii=False, default=str)

    print(f"\n    Registre exporté → {registry_path}")
    print(f"\n{'═'*62}")
    print(f"    NOTEBOOK 04 TERMINÉ — {len(TRANSFORMER_RESULTS)} modèle(s)")
    print(f"  Prêt pour : 05_evaluation_analysis.ipynb")
    print(f"{'═'*62}")

    Tous les imports OK
  Device : cpu
  Modèles actifs : ['camembert-base', 'distilbert-multilingual']
    Toutes les fonctions définies (A, B, C)
  Lancement de la boucle dans 3 secondes...

══════════════════════════════════════════════════════════════
  LANCEMENT — 2 modèle(s)
  Ordre   : camembert-base → distilbert-multilingual
  Device  : cpu
  HAS_GPU : False
══════════════════════════════════════════════════════════════


══════════════════════════════════════════════════════════════
  [1/2] CamemBERT-base
  HF      : camembert-base
  Output  : models/transformers/camembert-base/
══════════════════════════════════════════════════════════════

[1/9] Tokenizer ...
   Type    : CamembertTokenizer
   Vocab   : 32,005

[2/9] Datasets pré-tokenisés ...
   max_length = 128
   [train ] Tokenisation de 16,000 textes ...   (23.1s) | RAM≈16 Mo
   [val   ] Tokenisation de 5,000 textes ...   (6.8s) | RAM≈5 Mo
   [test  ] Tokenisation de 20,000 textes ...   (25.8s) | RAM≈20 Mo

[3/9] Modèle 

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 702.68it/s]
CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Co

   Params totaux   : 110,623,490
   Params entraîn. : 35,440,128 (32%)
   Couches gelées  : 120/201 (60%)
   Gain estimé     : ~30% de temps vs fine-tuning complet

[4/9] TrainingArguments ...
   fp16=False  lr=3e-05  epochs=1  batch=16

[5/9] Trainer + EarlyStopping ...

[6/9] Entraînement ...


Epoch,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall,Roc Auc
1,0.290727,0.246549,0.944300,0.944600,0.931000,0.957900,0.985900


Writing model shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.16s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.we

...

(2391.4s)

[8/9] Métriques finales (test — 20,000 ex.) ...

───────────────────────────────────────────────────────
              precision    recall  f1-score   support

     Négatif     0.9623    0.9246    0.9431     10408
     Positif     0.9215    0.9607    0.9407      9592

    accuracy                         0.9419     20000
   macro avg     0.9419    0.9426    0.9419     20000
weighted avg     0.9427    0.9419    0.9419     20000

   ROC AUC  : 0.9857
   Seuil    : 0.2788
   Temps    : 206.2 min

[9/9] Sauvegarde artefacts ...


Writing model shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.93s/it]


     Modèle + tokenizer
     y_pred / y_proba / y_true (.npy)
     threshold_analysis.parquet
     config.json + threshold.json

    CamemBERT-base — terminé en 16012.6s (266.9 min)
     F1=0.9407  AUC=0.9857  Seuil=0.2788


══════════════════════════════════════════════════════════════
  [2/2] DistilBERT-mlt
  HF      : distilbert-base-multilingual-cased
  Output  : models/transformers/distilbert-multilingual/
══════════════════════════════════════════════════════════════

[1/9] Tokenizer ...

    ERREUR — DistilBERT-mlt
     Type    : ConnectError
     Message : [Errno -3] Temporary failure in name resolution
     Traceback complet :


Traceback (most recent call last):
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/home/djoknone/miniconda3/envs/sentiment/lib/python3.10/site-packages/httpcore/_sync/connection.py", line 78, in handle_r



══════════════════════════════════════════════════════════════
  BOUCLE TERMINÉE
  Durée totale : 16013.6s  (266.9 min)
  Réussis      : 1/2
  Échoués      : ['distilbert-multilingual']
══════════════════════════════════════════════════════════════

───────────────────────────────────────────────────────
  RÉSUMÉ DES PERFORMANCES (test set)
───────────────────────────────────────────────────────
        Modèle     F1  Accuracy  Precision  Recall  ROC_AUC   Seuil  Temps_min
CamemBERT-base 0.9407    0.9419     0.9215  0.9607   0.9857 0.27876      206.2

───────────────────────────────────────────────────────
  VALIDATION DE COHÉRENCE
───────────────────────────────────────────────────────
     CamemBERT-base — métriques cohérentes

    Registre exporté → /home/djoknone/Bureau/MDSMS/MDSMS2/Text-Mining_Web-Mining/Projet_Examen/sentiment_allocine/models/transformers/transformer_registry.json

══════════════════════════════════════════════════════════════
    NOTEBOOK 04 TERMINÉ — 1 modèle